# Gemma 4 E2B-it — smoke test

Downloads the smallest Gemma 4 variant (`google/gemma-4-E2B-it`) and runs two prompts so we can see how the base model behaves on this machine before any LoRA work begins. Per [`../docs/models.md`](../docs/models.md) > Base model, this is the only Gemma 4 size WAVE ships (~2.3B effective params, 128k context, multimodal).

**This notebook is a developer-workstation smoke test. It is NOT the production path.** Production runs this same model quantized to INT4 via `transformers.js` + WebGPU in the browser (web demo) or LiteRT on-device (mobile).

## Environment

Dependencies live in [`environment.yml`](./environment.yml). Before running this notebook, create and activate the conda env from the `models/` folder:

```bash
conda env create -f environment.yml
conda activate wave-models
jupyter lab
```

What's installed:

- `pytorch` — inference backend (CUDA / MPS / CPU)
- `transformers` — model + tokenizer loaders, chat templating, generation
- `accelerate` — `device_map="auto"` placement
- `huggingface_hub` — auth for the gated Gemma repo

Make sure the Jupyter kernel for this notebook is the `wave-models` env (Cursor / Jupyter: kernel picker → `Python (wave-models)` or whichever name your install assigns).

## Hugging Face auth

Gemma weights are gated. Before the first run:

1. Open <https://huggingface.co/google/gemma-4-E2B-it> and click **Acknowledge license**.
2. Create a read token at <https://huggingface.co/settings/tokens>.
3. Run the cell below and paste the token when prompted (it caches to `~/.cache/huggingface/`, so you only do this once per machine).

If you already ran `huggingface-cli login` in your shell, you can skip this cell.

In [2]:
from huggingface_hub import login

login()

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"device={device}  dtype={dtype}")

device=mps  dtype=torch.float16


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-4-E2B-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto",
)
model.eval()

print(f"loaded {model_id}  params={sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

In [ ]:
def generate(messages, max_new_tokens: int = 200) -> str:
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    prompt_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
        )

    new_tokens = output_ids[0, prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 1. Generic warm-up

Confirms the model loaded and produces sane text.

In [ ]:
messages = [
    {"role": "user", "content": "Explain what urge surfing is in two sentences."},
]

print(generate(messages))

## 2. WAVE-style medication-aware acknowledgment

Mirrors the job of the future `lora-med-ack` adapter from [`../docs/models.md`](../docs/models.md) §LoRA-1. We feed the model the same kind of structured input the adapter will see (medication, status, intensity, trigger, time of day) and ask for a 2–3 sentence trauma-informed acknowledgment.

What we read out of this cell is the **base-model baseline** — it is what the adapter has to improve on.

In [ ]:
system_prompt = (
    "You are WAVE, a trauma-informed urge-surfing companion for people in "
    "recovery from substance use disorder. Generate a 2-3 sentence acknowledgment "
    "that reflects the patient's medication status and current craving. "
    "Tone: warm, grounded, never toxic-positivity, never shaming. "
    "Never tell the patient to start, stop, or change a medication. "
    "Do not name any specific substance the patient might use."
)

patient_context = (
    "Medication: Buprenorphine / Suboxone\n"
    "Status: on-time dose\n"
    "Craving intensity: 7/10\n"
    "Trigger: stress\n"
    "Time of day: late evening\n"
)

messages = [
    {"role": "user", "content": f"{system_prompt}\n\n{patient_context}"},
]

print(generate(messages))

## Notes

- **Production quantization.** The shipped runtime quantizes this base to INT4 (~1.5 GB on disk) and runs it via `transformers.js` + WebGPU in the browser. This notebook runs the model in native precision on the dev machine — quantization is the runtime-export step in [`../docs/model-training.md`](../docs/model-training.md) §6.1.
- **LoRA replacement.** Cell 2's behavior is what the `lora-med-ack` adapter is trained to specialize. Once that adapter exists, the same input goes through base + adapter and the output should hit the matrix in [`../PRD.md`](../PRD.md) > Medication-Aware Prompt Logic more reliably than base alone does here.
- **Crisis triage.** Per [`../docs/models.md`](../docs/models.md) > Not fine-tuned — base model only, crisis triage stays on this base with rule-based routing. No LoRA ships for that surface.